# Intro to GANs with PyTorch and MNIST

Build and train a simple Generative Adversarial Network (GAN) and investigate how generated images improve during training.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)


In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Using device:", device)


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
print("Training samples:", len(train_dataset))


In [ ]:
LATENT_DIM = 100


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
g_optimizer = optim.Adam(G.parameters(), lr=0.0002)
d_optimizer = optim.Adam(D.parameters(), lr=0.0002)


In [ ]:
def show_generated(generator, n=16, epoch=None):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n, LATENT_DIM).to(device)
        fake = generator(z).cpu().view(n, 28, 28)

    fig, axes = plt.subplots(4,4,figsize=(6,6))
    for img, ax in zip(fake, axes.flatten()):
        ax.imshow(img, cmap="gray")
        ax.axis("off")

    if epoch is not None:
        plt.suptitle(f"Epoch {epoch}")
    plt.show()
    generator.train()


In [ ]:
show_generated(G)

In [ ]:
EPOCHS = 10

g_losses = []
d_losses = []

for epoch in range(EPOCHS):
    for real_images, _ in loader:
        batch_size = real_images.size(0)
        real_images = real_images.view(batch_size, -1).to(device)

        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        z = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = G(z)

        d_real = D(real_images)
        d_fake = D(fake_images.detach())

        d_loss = (
            criterion(d_real, real_labels) +
            criterion(d_fake, fake_labels)
        )

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        z = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = G(z)

        g_loss = criterion(D(fake_images), real_labels)

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS}  D={d_loss.item():.4f}  G={g_loss.item():.4f}")
    show_generated(G, epoch=epoch+1)


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(g_losses,label="Generator")
plt.plot(d_losses,label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Loss")
plt.legend()
plt.grid(True)
plt.show()


## Student Experiment

Run the notebook three times using latent dimensions 20, 100, and 200. Record final losses and compare generated image quality.

## Reflection Questions

1. Why does the generator use random noise?
2. What role does the discriminator play?
3. When do digits become recognizable?
4. What is mode collapse?
5. What effect did latent dimension have on image diversity?
6. How does a GAN differ from a CNN classifier?